# Taller API + Power BI — Clima en ciudades del mundo (Open-Meteo)

**Objetivo:** consultar datos desde una API pública, transformarlos y estructurarlos, y exportarlos a un archivo CSV listo para construir un dashboard en Power BI.

**Fuente de datos:** [Open-Meteo](https://open-meteo.com/) — API pública de pronóstico del clima, sin autenticación ni API key.

**Flujo del notebook:**
1. Consulta a la API con `requests` (pronóstico de 7 días para una lista de ciudades)
2. Conversión de la respuesta JSON a `DataFrame`
3. Limpieza y transformación de datos
4. Generación del dataset final
5. Exportación a `.csv`


In [1]:
# Librerías necesarias
import requests
import pandas as pd
import numpy as np
from datetime import datetime

## 1. Lista de ciudades a consultar

Open-Meteo no tiene un endpoint de "todas las ciudades": se consulta por coordenadas geográficas.
Por eso se arma manualmente una lista de ciudades representativas de distintas regiones del mundo,
con su latitud, longitud, país y región (esto también nos da las categorías para los slicers del dashboard).

In [2]:
ciudades = [
    {"ciudad": "Bogota",        "pais": "Colombia",       "region": "Sudamerica",   "lat": 4.71,   "lon": -74.07},
    {"ciudad": "Medellin",      "pais": "Colombia",       "region": "Sudamerica",   "lat": 6.25,   "lon": -75.56},
    {"ciudad": "Barranquilla",  "pais": "Colombia",       "region": "Sudamerica",   "lat": 10.96,  "lon": -74.80},
    {"ciudad": "Ciudad de Mexico", "pais": "Mexico",      "region": "Norteamerica", "lat": 19.43,  "lon": -99.13},
    {"ciudad": "Buenos Aires",  "pais": "Argentina",      "region": "Sudamerica",   "lat": -34.60, "lon": -58.38},
    {"ciudad": "Sao Paulo",     "pais": "Brasil",         "region": "Sudamerica",   "lat": -23.55, "lon": -46.63},
    {"ciudad": "Lima",          "pais": "Peru",           "region": "Sudamerica",   "lat": -12.05, "lon": -77.04},
    {"ciudad": "Santiago",      "pais": "Chile",          "region": "Sudamerica",   "lat": -33.45, "lon": -70.65},
    {"ciudad": "Nueva York",    "pais": "Estados Unidos", "region": "Norteamerica", "lat": 40.71,  "lon": -74.01},
    {"ciudad": "Los Angeles",   "pais": "Estados Unidos", "region": "Norteamerica", "lat": 34.05,  "lon": -118.24},
    {"ciudad": "Toronto",       "pais": "Canada",         "region": "Norteamerica", "lat": 43.65,  "lon": -79.38},
    {"ciudad": "Madrid",        "pais": "Espana",         "region": "Europa",       "lat": 40.42,  "lon": -3.70},
    {"ciudad": "Londres",       "pais": "Reino Unido",    "region": "Europa",       "lat": 51.51,  "lon": -0.13},
    {"ciudad": "Paris",         "pais": "Francia",        "region": "Europa",       "lat": 48.85,  "lon": 2.35},
    {"ciudad": "Berlin",        "pais": "Alemania",       "region": "Europa",       "lat": 52.52,  "lon": 13.40},
    {"ciudad": "Tokio",         "pais": "Japon",          "region": "Asia",         "lat": 35.68,  "lon": 139.65},
    {"ciudad": "Pekin",         "pais": "China",          "region": "Asia",         "lat": 39.90,  "lon": 116.40},
    {"ciudad": "Dubai",         "pais": "Emiratos Arabes Unidos", "region": "Asia", "lat": 25.20,  "lon": 55.27},
    {"ciudad": "El Cairo",      "pais": "Egipto",         "region": "Africa",       "lat": 30.04,  "lon": 31.24},
    {"ciudad": "Nairobi",       "pais": "Kenia",          "region": "Africa",       "lat": -1.29,  "lon": 36.82},
    {"ciudad": "Sidney",        "pais": "Australia",      "region": "Oceania",      "lat": -33.87, "lon": 151.21},
]

print(f"Ciudades a consultar: {len(ciudades)}")

Ciudades a consultar: 21


## 2. Consulta a la API

Se hace una petición por ciudad al endpoint `/v1/forecast`, pidiendo variables diarias:
temperatura máxima y mínima, precipitación total y velocidad máxima del viento, para los próximos 7 días.

In [3]:
url = "https://api.open-meteo.com/v1/forecast"

registros = []

for c in ciudades:
    params = {
        "latitude": c["lat"],
        "longitude": c["lon"],
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "timezone": "auto",
        "forecast_days": 7,
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()  # lanza un error si la petición falla (ej. status 4xx/5xx)

    data = response.json()
    diario = data["daily"]

    # 'diario' trae arrays paralelos: una posición por cada uno de los 7 días
    for i, fecha in enumerate(diario["time"]):
        registros.append({
            "ciudad": c["ciudad"],
            "pais": c["pais"],
            "region": c["region"],
            "fecha": fecha,
            "temp_max_c": diario["temperature_2m_max"][i],
            "temp_min_c": diario["temperature_2m_min"][i],
            "precipitacion_mm": diario["precipitation_sum"][i],
            "viento_max_kmh": diario["windspeed_10m_max"][i],
        })

print(f"Registros obtenidos: {len(registros)}  (≈ {len(ciudades)} ciudades x 7 días)")

Registros obtenidos: 147  (≈ 21 ciudades x 7 días)


## 3. De JSON a DataFrame

In [4]:
df = pd.DataFrame(registros)
df.head(10)

,ciudad,pais,region,fecha,temp_max_c,temp_min_c,precipitacion_mm,viento_max_kmh
0,Bogota,Colombia,Sudamerica,2026-08-31,20.6,12.1,1.7,17.7
1,Bogota,Colombia,Sudamerica,2026-09-01,21.6,10.8,1.0,13.8
2,Bogota,Colombia,Sudamerica,2026-09-02,20.4,10.7,1.5,17.9
3,Bogota,Colombia,Sudamerica,2026-09-03,20.5,10.4,1.6,15.2
4,Bogota,Colombia,Sudamerica,2026-09-04,20.4,11.4,0.9,14.2
5,Bogota,Colombia,Sudamerica,2026-09-05,20.8,10.9,0.9,14.9
6,Bogota,Colombia,Sudamerica,2026-09-06,20.9,10.1,2.4,12.5
7,Medellin,Colombia,Sudamerica,2026-08-31,29.9,17.2,2.7,6.1
8,Medellin,Colombia,Sudamerica,2026-09-01,27.6,17.4,4.3,7.1
9,Medellin,Colombia,Sudamerica,2026-09-02,28.8,17.0,3.2,11.3


## 4. Exploración inicial

Antes de limpiar, se revisa la forma del dataset: tipos de dato, nulos y duplicados.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ciudad            147 non-null    object 
 1   pais              147 non-null    object 
 2   region            147 non-null    object 
 3   fecha             147 non-null    object 
 4   temp_max_c        147 non-null    float64
 5   temp_min_c        147 non-null    float64
 6   precipitacion_mm  147 non-null    float64
 7   viento_max_kmh    147 non-null    float64
dtypes: float64(4), object(4)
memory usage: 9.3+ KB


In [6]:
# Conteo de valores nulos por columna
df.isnull().sum()

ciudad              0
pais                0
region              0
fecha               0
temp_max_c          0
temp_min_c          0
precipitacion_mm    0
viento_max_kmh      0
dtype: int64

## 5. Limpieza y transformación

- Se eliminan registros sin temperatura (dato clave para el análisis).
- Se convierte la columna `fecha` a tipo fecha real.
- Se crean dos columnas nuevas útiles para el dashboard:
  - `rango_termico`: diferencia entre temperatura máxima y mínima del día.
  - `condicion`: categoría "Lluvioso" o "Seco", según si hubo precipitación relevante (>1 mm) — esta será uno de los slicers.
- Se eliminan duplicados, si los hubiera.

In [7]:
# Copia de trabajo para no modificar el DataFrame original
df_limpio = df.copy()

# Eliminar registros sin datos clave de temperatura
df_limpio = df_limpio.dropna(subset=["temp_max_c", "temp_min_c"])

# Convertir a tipo fecha
df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"])

# Nueva métrica: rango térmico del día
df_limpio["rango_termico"] = (df_limpio["temp_max_c"] - df_limpio["temp_min_c"]).round(1)

# Nueva categoría: condición climática (para usar como slicer)
df_limpio["condicion"] = np.where(df_limpio["precipitacion_mm"] > 1, "Lluvioso", "Seco")

# Eliminar duplicados (misma ciudad + misma fecha)
df_limpio = df_limpio.drop_duplicates(subset=["ciudad", "fecha"])

# Ordenar columnas y resetear el índice
df_limpio = df_limpio[[
    "ciudad", "pais", "region", "fecha",
    "temp_max_c", "temp_min_c", "rango_termico",
    "precipitacion_mm", "viento_max_kmh", "condicion"
]].sort_values(["region", "pais", "ciudad", "fecha"]).reset_index(drop=True)

print(f"Filas finales: {len(df_limpio)}")
df_limpio.head(10)

Filas finales: 147


,ciudad,pais,region,fecha,temp_max_c,temp_min_c,rango_termico,precipitacion_mm,viento_max_kmh,condicion
0,El Cairo,Egipto,Africa,2026-09-01,35.7,22.9,12.8,0.0,11.6,Seco
1,El Cairo,Egipto,Africa,2026-09-02,35.6,23.7,11.9,0.0,14.0,Seco
2,El Cairo,Egipto,Africa,2026-09-03,36.5,23.2,13.3,0.0,13.6,Seco
3,El Cairo,Egipto,Africa,2026-09-04,36.5,23.7,12.8,0.0,10.5,Seco
4,El Cairo,Egipto,Africa,2026-09-05,36.5,23.1,13.4,0.0,13.7,Seco
5,El Cairo,Egipto,Africa,2026-09-06,35.5,23.2,12.3,0.0,13.6,Seco
6,El Cairo,Egipto,Africa,2026-09-07,38.0,23.1,14.9,0.0,13.3,Seco
7,Nairobi,Kenia,Africa,2026-09-01,23.1,14.0,9.1,1.9,11.3,Lluvioso
8,Nairobi,Kenia,Africa,2026-09-02,26.4,14.9,11.5,1.1,14.4,Lluvioso
9,Nairobi,Kenia,Africa,2026-09-03,27.1,14.2,12.9,0.1,16.4,Seco


## 6. Dataset final

Vista general del dataset ya limpio y listo para exportar.

In [8]:
df_limpio.describe(include="all")

,ciudad,pais,region,fecha,temp_max_c,temp_min_c,rango_termico,precipitacion_mm,viento_max_kmh,condicion
count,147,147,147,147,147.000000,147.000000,147.000000,147.000000,147.000000,147
unique,21,18,6,NaN,NaN,NaN,NaN,NaN,NaN,2
top,El Cairo,Colombia,Sudamerica,NaN,NaN,NaN,NaN,NaN,NaN,Seco
freq,7,21,49,NaN,NaN,NaN,NaN,NaN,NaN,99
mean,NaN,NaN,NaN,2026-09-03 11:25:42.857142784,26.706122,17.525170,9.180952,3.126395,13.545578,NaN
min,NaN,NaN,NaN,2026-08-31 00:00:00,10.400000,2.300000,1.200000,0.000000,4.200000,NaN
25%,NaN,NaN,NaN,2026-09-02 00:00:00,21.900000,13.600000,6.650000,0.000000,10.100000,NaN
50%,NaN,NaN,NaN,2026-09-03 00:00:00,26.100000,17.600000,8.700000,0.000000,13.600000,NaN
75%,NaN,NaN,NaN,2026-09-05 00:00:00,30.150000,21.100000,11.200000,2.200000,16.400000,NaN
max,NaN,NaN,NaN,2026-09-07 00:00:00,41.400000,32.500000,19.100000,71.800000,29.900000,NaN


## 7. Exportación a CSV

Se exporta el dataset final, listo para cargarlo en Power BI.

In [9]:
df_limpio.to_csv("clima_ciudades_dataset.csv", index=False, encoding="utf-8-sig")
print("Archivo 'clima_ciudades_dataset.csv' generado correctamente.")

Archivo 'clima_ciudades_dataset.csv' generado correctamente.


## Notas para el dashboard en Power BI

Con este dataset se pueden construir, por ejemplo:

- **Slicers:** `region`, `pais` (o `ciudad`), `condicion`
- **Visualizaciones:**
  - Temperatura máxima promedio por ciudad (barras)
  - Evolución de la temperatura a lo largo de los 7 días (línea)
  - Precipitación total por región (barras o mapa)
- **Insight sugerido:** comparar cómo varían temperatura y precipitación entre regiones del mundo en la misma semana, destacando qué ciudades tendrán condiciones más lluviosas o más cálidas.
